In [ ]:
from datetime import date
import math
from pathlib import Path
from collections import defaultdict
import spacy
from spacytextblob.spacytextblob import SpacyTextBlob
import re
import webbrowser
from pytesseract import pytesseract
import json
import cv2
from transformers import AutoTokenizer, AutoModelWithLMHead
import seaborn as sns
from matplotlib import pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import statsmodels.api as sm
from wordcloud import WordCloud
from openai import OpenAI

# Configuration variables
g_keywords = ["chinese", "japanese", "korean", "okinawan", "taiwanese", "tibetan", "\"east+asian\"", "oriental", "chinaman", "chinamen", "jap", "chink", "coolie", "celestial"]
g_search_date_range_start = date(1850, 1, 1)
g_search_date_range_end = date(2024, 7, 9)
g_num_articles_per_time_period = 20
g_time_block_range_years = 5

g_search_date_range_days = (g_search_date_range_end - g_search_date_range_start).days
g_num_time_blocks = math.ceil(g_search_date_range_days / (365 * g_time_block_range_years))
g_emotions = ['anger', 'fear', 'sadness', 'joy', 'love', 'surprise']

print("Number of time blocks: " + str(g_num_time_blocks))
print("Number of articles to download: " + str(g_num_time_blocks * g_num_articles_per_time_period))

def create_directories_if_do_not_exist(directories):
    for directory in directories:
        directory_path = Path.cwd() / directory
        if not directory_path.exists():
            directory_path.mkdir()
            print("Created directory: \"" + directory + "\"")

g_storage_directories = ["articles", "contexts", "uncorrected_contexts", "results"]
create_directories_if_do_not_exist(g_storage_directories)

g_time_block_number = int(input("Pick time block, 0-" + str(g_num_time_blocks - 1) + ": "))
if g_time_block_number < 0 or g_time_block_number >= g_num_time_blocks:
    print("Invalid time block!")
    exit()


def get_starting_date(time_block_num, search_date_range_start, time_block_range_years):
    return date(search_date_range_start.year + time_block_num * time_block_range_years, 1, 1)


def get_ending_date(time_block_num, search_date_range_start, search_date_range_end, time_block_range_years):
    ending_date = date(search_date_range_start.year + (time_block_num + 1) * time_block_range_years - 1, 12, 31)
    if ending_date > search_date_range_end:
        return search_date_range_end
    else:
        return ending_date


g_start_date = get_starting_date(g_time_block_number, g_search_date_range_start, g_time_block_range_years)
g_end_date = get_ending_date(g_time_block_number, g_search_date_range_start, g_search_date_range_end, g_time_block_range_years)

print("Start date: " + g_start_date.strftime('%Y-%m-%d'))
print("End date: " + g_end_date.strftime('%Y-%m-%d'))
print("Length of time period in days: " + str((g_end_date - g_start_date).days))

Load functions

In [ ]:
def parse_time_block_str(time_blocks_str):
    unprocessed_time_block_str_list = time_blocks_str.split(", ")
    time_block_list = []
    for time_block_str in unprocessed_time_block_str_list:
        if "-" in time_block_str:
            time_block_range = time_block_str.split("-")
            time_block_list.extend(range(int(time_block_range[0]), int(time_block_range[1]) + 1))
        else:
            time_block_list.append(int(time_block_str))
    return time_block_list


def get_time_block_number(year, search_date_range_start, time_block_range_years):
    return math.floor((year - search_date_range_start.year) / time_block_range_years)


def get_keywords_with_num_hits(keywords, start_date, end_date):
    keywords_with_num_hits = {}
    for keyword in keywords:
        webbrowser.open("https://www.newspapers.com/search/results/?country=us&date-end=" + end_date.strftime('%Y-%m-%d') + "&date-start=" + start_date.strftime('%Y-%m-%d') + "&entity-types=page&keyword=" + keyword)
        keywords_with_num_hits[keyword] = int(input("Number of hits: ").replace(",", ""))
    return keywords_with_num_hits


def get_keywords_with_num_articles(keywords_with_num_hits, num_articles_per_time_period):
    total_hits = sum(keywords_with_num_hits.values())
    article_remainders = []
    keywords_with_num_articles = {}
    for keyword, hits in keywords_with_num_hits.items():
        keywords_with_num_articles[keyword] = int((hits / total_hits) * num_articles_per_time_period)
        article_remainders.append((keyword, ((hits / total_hits) * num_articles_per_time_period) % 1))
    article_remainders.sort(key=lambda keyword_with_article_remainder: keyword_with_article_remainder[1], reverse=True)
    num_searches_short = num_articles_per_time_period - sum(keywords_with_num_articles.values())
    for i in range(num_searches_short):
        keywords_with_num_articles[article_remainders[i][0]] += 1
    return keywords_with_num_articles


def get_keywords_with_urls_and_num_articles(keywords_with_num_articles, start_date, end_date):
    keywords_with_urls_and_num_articles = []
    for keyword, num_articles in keywords_with_num_articles.items():
        keywords_with_urls_and_num_articles.append({
            "keyword": keyword,
            "url": "https://www.newspapers.com/search/results/?country=us&date-end=" + end_date.strftime('%Y-%m-%d') + "&date-start=" + start_date.strftime('%Y-%m-%d') + "&entity-types=page&keyword=" + keyword,
            "num_articles": num_articles
        })
    return keywords_with_urls_and_num_articles


def run_article_download_helper(keywords_with_urls_and_num_articles):
    for keyword_with_urls_and_num_articles in keywords_with_urls_and_num_articles:
        if keyword_with_urls_and_num_articles["num_articles"] != 0:
            text_input = input("Please download " + str(keyword_with_urls_and_num_articles["num_articles"]) + " articles with the keyword \"" + keyword_with_urls_and_num_articles["keyword"] + "\". Press return to continue. Type \"stop\" to stop.")
            if text_input == "stop":
                break
            webbrowser.open(keyword_with_urls_and_num_articles["url"])
            text_input = input("Once you have downloaded " + str(keyword_with_urls_and_num_articles["num_articles"]) + " articles with the keyword \"" + keyword_with_urls_and_num_articles["keyword"] + "\", press return to continue. Type \"stop\" to stop.")
            if text_input == "stop":
                break


def get_date_from_string(date_str):
    date_parts = date_str.split('-')
    return date(int(date_parts[0]), int(date_parts[1]), int(date_parts[2]))


def get_info_from_file_stem(file_stem):
    parts = file_stem.split('_')
    newspaper_name_parts = []

    for part in parts:
        if part.isdigit() and len(part) == 4:
            year_index = parts.index(part)
            break
        newspaper_name_parts.append(part)

    newspaper_name = ' '.join(newspaper_name_parts)
    date_published = date(int(parts[year_index]), int(parts[year_index + 1]), int(parts[year_index + 2]))

    return newspaper_name, date_published


def write_time_period_sorted_info_to_files(contexts, target_dir_path):
    if not target_dir_path.exists():
        target_dir_path.mkdir()
    for time_period, time_period_info in contexts.items():
        with open(target_dir_path / f"{time_period}.json", 'w+') as time_period_info_file:
            json.dump(time_period_info, time_period_info_file, indent=4, ensure_ascii=False)


def date_to_years_with_decimal(date_to_convert):
    start_of_year = date(date_to_convert.year, 1, 1)
    start_of_next_year = date(date_to_convert.year + 1, 1, 1)
    year_length = (start_of_next_year - start_of_year).days
    day_of_year = (date_to_convert - start_of_year).days + 1

    return date_to_convert.year + (day_of_year - 1) / year_length


def get_values_for_all_articles(info_type, results_dir_path):
    times = []
    values = []
    for result_by_time_period_path in results_dir_path.rglob("*"):
        if result_by_time_period_path.suffix == ".json":
            with open(result_by_time_period_path) as result_file:
                result_by_time_period = json.load(result_file)
                for keyword, keyword_contexts in result_by_time_period.items():
                    for keyword_context in keyword_contexts:
                        times.append(keyword_context['date'])
                        values.append(keyword_context[info_type])
    return times, values


def get_all_values_by_time_period(info_type, results_dir_path):
    all_values_by_time_period = [None] * g_num_time_blocks
    for result_by_time_period_path in results_dir_path.rglob("*"):
        if result_by_time_period_path.suffix == ".json":
            time_period = int(result_by_time_period_path.stem)
            with open(result_by_time_period_path) as result_file:
                result_by_time_period = json.load(result_file)
                values = []
                for keyword, keyword_contexts in result_by_time_period.items():
                    for keyword_context in keyword_contexts:
                        values.append(keyword_context[info_type])
                all_values_by_time_period[time_period] = values
    return all_values_by_time_period


def get_mean_by_time_period(info_type, results_dir_path):
    mean_by_time_period = [0] * g_num_time_blocks
    for result_by_time_period_path in results_dir_path.rglob("*"):
        if result_by_time_period_path.suffix == ".json":
            time_period = int(result_by_time_period_path.stem)
            with open(result_by_time_period_path) as result_file:
                result_by_time_period = json.load(result_file)
                values = []
                for keyword, keyword_contexts in result_by_time_period.items():
                    for keyword_context in keyword_contexts:
                        values.append(keyword_context[info_type])
                mean_by_time_period[time_period] = sum(values) / len(values)
    return mean_by_time_period


def add_frequency_dictionaries_excluding_keywords(dict1, dict2):
        combined_dict = dict(dict1)
        for key, value in dict2.items():
            key_in_global_keywords = False
            for g_keyword in g_keywords:
                if g_keyword in key:
                    key_in_global_keywords = True
                    break
            if not key_in_global_keywords:
                if key in combined_dict:
                    combined_dict[key] += value
                else:
                    combined_dict[key] = value
        return combined_dict


def get_combined_frequencies_by_time_period_excluding_keywords(info_type, results_dir_path):
    combined_frequency_by_time_period = defaultdict(int)
    for result_by_time_period_path in results_dir_path.rglob("*"):
        if result_by_time_period_path.suffix == ".json":
            time_period = result_by_time_period_path.stem
            with open(result_by_time_period_path) as result_file:
                result_by_time_period = json.load(result_file)
                combined_frequency = defaultdict(int)
                for keyword, keyword_contexts in result_by_time_period.items():
                    for keyword_context in keyword_contexts:
                        if isinstance(keyword_context[info_type], str) and keyword_context[info_type] not in g_keywords:
                            combined_frequency[keyword_context[info_type]] += 1
                        else:
                            combined_frequency = add_frequency_dictionaries_excluding_keywords(combined_frequency, keyword_context[info_type])
                combined_frequency_by_time_period[time_period] = combined_frequency
    return combined_frequency_by_time_period


def get_emotion_color(emotion):
    emotion_colors = {'anger': 'r', 'fear': '#2e2d2d', 'sadness': 'c', 'joy': 'g', 'love': 'm', 'surprise': 'y'}
    return emotion_colors[emotion]


def get_emotion_frequencies_dataframe(emotions, results_dir_path):
    emotion_frequencies = [None] * g_num_time_blocks
    for time_period in range(g_num_time_blocks):
        emotion_frequency_for_time_period = [0] * 6
        if (results_dir_path / f"{time_period}.json").is_file():
            with open(results_dir_path / f"{time_period}.json") as time_period_file:
                time_period_info = json.load(time_period_file)
                for keyword, keyword_contexts in time_period_info.items():
                    for keyword_context in keyword_contexts:
                        emotion = keyword_context['emotion']
                        emotion_frequency_for_time_period[emotions.index(emotion)] += 1
        emotion_frequencies[time_period] = emotion_frequency_for_time_period
    return pd.DataFrame(emotion_frequencies, columns=emotions)


def generate_word_clouds(word_frequencies_by_time_period):
    for time_period, word_frequencies in word_frequencies_by_time_period.items():
        print("Time period: " + time_period)
        word_cloud = WordCloud().generate_from_frequencies(word_frequencies)
        plt.imshow(word_cloud, interpolation='bilinear')
        plt.axis("off")
        plt.show()

Get the number of articles needed for each keyword by pasting in the number of hits for each keyword

In [ ]:
g_keywords_with_num_articles = get_keywords_with_num_articles(get_keywords_with_num_hits(g_keywords, g_start_date, g_end_date), g_num_articles_per_time_period)
print(g_keywords_with_num_articles)

Generate the URLs to grab the articles from. Note: number of URLs is not necessarily equal to number of articles to download.

In [ ]:
g_keywords_with_urls_and_num_articles = get_keywords_with_urls_and_num_articles(g_keywords_with_num_articles, g_start_date, g_end_date)
print("Generated " + str(len(g_keywords_with_urls_and_num_articles)) + " URLs")

Utility to guide users in downloading articles. Place articles in a folder with the same name as the keyword, which will be in the "articles" folder. Name the file with the date in the format "YYYY-MM-DD"

In [ ]:
run_article_download_helper(g_keywords_with_urls_and_num_articles)

Load ML

In [ ]:
nlp_sentence_segmentation = spacy.load("en_core_web_trf")
nlp_sentiment_analysis = spacy.load("en_core_web_trf")
nlp_sentiment_analysis.add_pipe("spacytextblob")

tokenizer = AutoTokenizer.from_pretrained("mrm8488/t5-base-finetuned-emotion")
model = AutoModelWithLMHead.from_pretrained("mrm8488/t5-base-finetuned-emotion")

client = OpenAI()

# Taken from https://huggingface.co/mrm8488/t5-base-finetuned-emotion
def get_emotion(text):
  input_ids = tokenizer.encode(text + '</s>', return_tensors='pt')
  output = model.generate(input_ids=input_ids, max_length=2)
  dec = [tokenizer.decode(ids) for ids in output]
  label = dec[0]
  return str(label).replace("<pad> ", "")


def get_contexts_for_keyword_from_text(text, keyword):
    possible_contexts = []
    sentences = list(nlp_sentence_segmentation(text).sents)
    for num in range(len(sentences) - 2):
        sentence = sentences[num + 1]
        if keyword in sentence.text.lower():
            possible_contexts.append(sentences[num].text + " " + sentence.text + " " + sentences[num + 2].text)
    possible_contexts.sort(reverse=True, key=len)
    return possible_contexts[0]


def get_and_write_all_contexts_for_keywords(time_period_list, articles_dir_path, uncorrected_contexts_dir_path):
    for time_period in time_period_list:
        keywords_with_contexts_for_time_period = {}
        counter = 0
        time_period_dir_path = articles_dir_path / str(time_period)
        for articles_by_keyword_dir_path in time_period_dir_path.rglob("*"):
            if articles_by_keyword_dir_path.is_dir():
                keyword = articles_by_keyword_dir_path.name
                keywords_with_contexts_for_time_period[keyword] = []
                for article_path in articles_by_keyword_dir_path.rglob("*"):
                    if article_path.is_file() and article_path.stem != ".DS_Store":
                        newspaper_name, date = get_info_from_file_stem(article_path.stem)
                        img = cv2.imread(article_path, cv2.IMREAD_GRAYSCALE)
                        blur = cv2.GaussianBlur(img,(5, 5), 0)
                        th3 = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
                        text = pytesseract.image_to_string(th3)
                        cleaned_text = cleaned_text = re.sub(r'\s+', ' ', text.replace(" |", "").replace("|", "").replace("-\n", "").replace("\n", " "))
                        keywords_with_contexts_for_time_period[keyword].append({
                            "newspaper": newspaper_name,
                            "date": date.strftime('%Y-%m-%d'),
                            "keyword": keyword,
                            "text": cleaned_text
                        })
                        counter += 1
        write_time_period_sorted_info_to_files({ time_period: keywords_with_contexts_for_time_period }, uncorrected_contexts_dir_path)
        print(f"{time_period}: {counter} articles")


def narrow_contexts(time_period_list, uncorrected_contexts_dir_path, contexts_dir_path):
    for time_period in time_period_list:
        keywords_with_contexts_for_time_period = {}
        with open(uncorrected_contexts_dir_path / f"{time_period}.json") as uncorrected_contexts_file:
            keywords_with_contexts = json.load(uncorrected_contexts_file)
            for keyword, contexts in keywords_with_contexts.items():
                keywords_with_contexts_for_time_period[keyword] = []
                for context in contexts:
                    keywords_with_contexts_for_time_period[keyword].append({
                        "newspaper": context["newspaper"],
                        "date": context["date"],
                        "keyword": context["keyword"],
                        "text": get_contexts_for_keyword_from_text(context["text"], keyword)
                    })
        write_time_period_sorted_info_to_files({ time_period: keywords_with_contexts_for_time_period }, contexts_dir_path)


def run_analysis_from_contexts_and_write(time_period_list, contexts_dir_path, results_dir_path):
    for time_period in time_period_list:
        keywords_with_contexts_and_analysis_for_time_period = {}
        with open(contexts_dir_path / f"{time_period}.json") as contexts_file:
            keywords_with_contexts = json.load(contexts_file)
            for keyword, contexts in keywords_with_contexts.items():
                keywords_with_contexts_and_analysis_for_time_period[keyword] = []
                for context in contexts:
                    doc = nlp_sentiment_analysis(context['text'])
                    word_frequency = defaultdict(int)
                    for token in doc:
                        if token.is_alpha and not token.is_stop:
                            word_frequency[token.text.lower()] += 1
                    context['polarity'] = doc._.blob.polarity
                    context['subjectivity'] = doc._.blob.subjectivity
                    context['word_frequency'] = word_frequency
                    context['emotion'] = get_emotion(context['text'])
                    keywords_with_contexts_and_analysis_for_time_period[keyword].append(context)
        write_time_period_sorted_info_to_files({ time_period: keywords_with_contexts_and_analysis_for_time_period }, results_dir_path)

Get contexts surrounding keywords with OCR and sentence segmentation. Provide a list of numbers (e.g. 1, 2, 3) and/or a range (e.g. 10-15). Write to file in all_contexts

In [ ]:
g_articles_dir_path = Path.cwd() / 'articles'
get_and_write_all_contexts_for_keywords(parse_time_block_str(input("Enter time periods. Provide a list of numbers (e.g. 1, 2, 3) and/or a range (e.g. 10-15).")), g_articles_dir_path, Path.cwd() / 'uncorrected_contexts')

Remove non-context after manual corrections

Read in contexts from JSON file for analysis, perform and output to results

In [ ]:
run_analysis_from_contexts_and_write(parse_time_block_str(input("Enter time periods for sentiment analysis. Provide a list of numbers (e.g. 1, 2, 3) and/or a range (e.g. 10-15).")), Path.cwd() / 'contexts', Path.cwd() / 'results')

Compile and display data

In [ ]:
include_titles = input("Include titles? (y/n): ") == "y"

linear_regressions_table = {}

mean_polarity_by_time_period = get_mean_by_time_period('polarity', Path.cwd() / 'results')

bar_start_dates = [get_starting_date(time_block_num, g_search_date_range_start, g_time_block_range_years).year for time_block_num in range(g_num_time_blocks)]

date_ticks = list(range(math.floor(g_search_date_range_start.year / 10) * 10, math.ceil(g_search_date_range_end.year / 10) * 10 + 10, 10))
polarity_bar_y_ticks = [tick / 100 for tick in range(-12, 14, 2)]
polarity_bar_y_lim = (polarity_bar_y_ticks[0], polarity_bar_y_ticks[-1])

polarity_avg_bar_fig, polarity_avg_bar_ax = plt.subplots(figsize=(12, 9))
polarity_avg_bar_ax.bar(bar_start_dates, mean_polarity_by_time_period, color='b', edgecolor='k', linewidth=1, width=5, align='edge')
polarity_avg_bar_ax.set(xlabel='Time (years)', ylabel='Sentiment Polarity', title='mean Sentiment Polarity by Time Period (Bar Chart)' if include_titles else '', xticks=date_ticks, yticks=polarity_bar_y_ticks, ylim=polarity_bar_y_lim)

plt.axhline(y=0, color='k')

plt.show()

print("Min mean polarity: " + str(min(mean_polarity_by_time_period)) + " for " + str(mean_polarity_by_time_period.index(min(mean_polarity_by_time_period))))
print("Max mean polarity: " + str(max(mean_polarity_by_time_period)) + " for " + str(mean_polarity_by_time_period.index(max(mean_polarity_by_time_period))))
mean_polarity_overall = sum(mean_polarity_by_time_period) / len(mean_polarity_by_time_period)
print("Mean mean polarity: " + str(mean_polarity_overall))
print("Mean absolute deviation of mean polarity: " + str(sum([abs(mean_polarity_overall - mean_polarity_val) for mean_polarity_val in mean_polarity_by_time_period]) / len(mean_polarity_by_time_period)))

article_date_strs, article_polarities = get_values_for_all_articles('polarity', Path.cwd() / 'results')
article_date_years = [date_to_years_with_decimal(get_date_from_string(date_str)) for date_str in article_date_strs]

polarity_full_range_y_ticks = [tick / 10 for tick in range(-10, 11, 1)]

mean_polarity_linear_regression = LinearRegression().fit(np.array(article_date_years).reshape(-1, 1), np.array(article_polarities).reshape(-1, 1))
mean_polarity_pred = mean_polarity_linear_regression.predict(np.array(article_date_years).reshape(-1, 1))
mean_polarity_stats = sm.OLS(pd.DataFrame({"Mean sentiment polarity": article_polarities}), sm.add_constant(pd.DataFrame({"year": article_date_years}))).fit()
linear_regressions_table["Sentiment polarity"] = [mean_polarity_stats.params.loc['year'], mean_polarity_stats.rsquared, mean_polarity_stats.pvalues.loc['year']]

polarity_scatter_fig, polarity_scatter_ax = plt.subplots(figsize=(12, 9))
polarity_scatter_ax.scatter(article_date_years, article_polarities, color='b', edgecolor='k', linewidth=1)
polarity_scatter_ax.set(xlabel='Time (years)', ylabel='Sentiment Polarity', title='Sentiment Polarity by Time Period (Scatter Plot)' if include_titles else '', xticks=date_ticks, yticks=polarity_full_range_y_ticks, ylim=(-1, 1))
polarity_scatter_ax.plot(article_date_years, mean_polarity_pred, color='r', linewidth=2)

plt.show()

polarities_for_time_periods = get_all_values_by_time_period('polarity', Path.cwd() / 'results')

polarity_box_fig, polarity_box_ax = plt.subplots(figsize=(12, 9))
polarity_box_ax.boxplot(polarities_for_time_periods, widths=1)
polarity_box_ax.set(xlabel='Time (years)', ylabel='Sentiment Polarity', title='Sentiment Polarity by Time Period (Box Plot)' if include_titles else '', yticks=polarity_full_range_y_ticks, ylim=(-1, 1))
polarity_box_ax.set_xticks([tick / 10 for tick in range(5, (g_num_time_blocks + 2) * 10 + 5, 20)], labels=date_ticks)

plt.show()

combined_emotion_frequencies_by_time_period = get_emotion_frequencies_dataframe(g_emotions, Path.cwd() / 'results')

emotion_frequencies_overall = {}
for column in combined_emotion_frequencies_by_time_period:
    emotion_frequencies_overall[column] = combined_emotion_frequencies_by_time_period[column].sum()

print("Negative emotions: " + str(sum([emotion_frequencies_overall[emotion] for emotion in ['anger', 'fear', 'sadness']]) / sum(emotion_frequencies_overall.values()) * 100))
print("Positive emotions: " + str(sum([emotion_frequencies_overall[emotion] for emotion in ['joy', 'love']]) / sum(emotion_frequencies_overall.values()) * 100))

patches = list(reversed([mpatches.Patch(color=get_emotion_color(emotion), label=emotion) for emotion in g_emotions]))

emotion_pie_fig, emotion_pie_ax = plt.subplots(figsize=(12, 9))
emotion_pie_ax.pie(emotion_frequencies_overall.values(), autopct=lambda pct: ('%1.1f%%' % pct) if pct > 1 else '', pctdistance=1.1, startangle=90, colors=[get_emotion_color(emotion) for emotion in emotion_frequencies_overall.keys()])
emotion_pie_ax.legend(handles=patches, loc='upper right', bbox_to_anchor=(1.13, 1.0)).get_frame().set_linewidth(2)

article_frequency_y_ticks = range(0, 22, 2)
bottom = np.zeros(35)

article_frequency_fig, article_frequency_ax = plt.subplots(figsize=(12, 9))

for emotion, frequencies in combined_emotion_frequencies_by_time_period.items():
    p = article_frequency_ax.bar(bar_start_dates, frequencies, color=get_emotion_color(emotion), edgecolor='k', linewidth=0.75, width=5, label=emotion, bottom=bottom, align='edge')
    bottom += frequencies
    emotion_frequency_stats = sm.OLS(pd.DataFrame({"Emotion Frequency": frequencies}), sm.add_constant(pd.DataFrame({"year": bar_start_dates}))).fit()
    linear_regressions_table[f"Articles with {emotion} as dominant emotion"] = [emotion_frequency_stats.params.loc['year'], emotion_frequency_stats.rsquared, emotion_frequency_stats.pvalues.loc['year']]


article_frequency_ax.set(xlabel='Time (years)', ylabel='Articles', title='Article Emotion Frequency by Time Period' if include_titles else '', xticks=date_ticks, yticks=article_frequency_y_ticks)
article_frequency_ax.legend(handles=patches, loc='upper right', bbox_to_anchor=(1.13, 1.0)).get_frame().set_linewidth(2)

plt.show()

print(pd.DataFrame.from_dict(linear_regressions_table, orient='index', columns=['Coefficient', 'R^2', 'P-value']).to_latex())

combined_word_frequencies_by_time_period = get_combined_frequencies_by_time_period_excluding_keywords('word_frequency', Path.cwd() / 'results')
generate_word_clouds(combined_word_frequencies_by_time_period)